**Import necessary libraries**

In [ ]:
# Code Cell 0: Imports
# These are all the necessary libraries for the script.
# - pandas: For reading and manipulating Excel files and data.
# - os: For handling file path operations (though less used with direct GCS paths).
# - google.colab.auth: For authenticating the user in Colab to access Google Cloud resources.
# - google.cloud.storage: The client library to interact with Google Cloud Storage (GCS).

import pandas as pd
import datetime # For datetime.datetime.now()
import os
import subprocess # For running gcloud shell commands
import re # For regular expressions used in sanitizing column names
from google.colab import auth
from google.cloud import storage
from google.cloud import bigquery # The client library to interact with Google BigQuery.


# It's good practice to authenticate early if needed.
# In Colab Enterprise, the service account might have default permissions.
# If you encounter permission errors, uncomment and run the line below in a separate cell or here.
# auth.authenticate_user()

print("Libraries imported successfully!")
print(f"Pandas version: {pd.__version__}")


**Environment setup**

In [ ]:
# Code Cell 1: Environment Setup & User Configuration
# -----------------------------------------------------------------------------
# Set environment variables and user-configurable parameters here.
# Run this cell to verify the output before proceeding.
# -----------------------------------------------------------------------------

# --- Environment and Execution Information ---
# Change to "us", "eu", etc. as per your requirements for location context if needed elsewhere.
location = "us"

# Get the current date and time
now = datetime.datetime.now()

# Format the date and time as desired (e.g., for logging or unique naming)
formatted_date = now.strftime("%Y-%m-%d-%H-%M-%S") # Added seconds for more uniqueness

# Get some values using gcloud and environment variables
# These assume the script is run in a Google Cloud environment (e.g., Colab Enterprise, Vertex AI Workbench)
try:
    project_id = os.environ["GOOGLE_CLOUD_PROJECT"]
    # The `!` prefix executes shell commands in Colab. Output is a list of strings.
    user_output = !(gcloud auth list --filter=status:ACTIVE --format="value(account)")

    if len(user_output) == 1:
        user = user_output[0]
    elif len(user_output) > 1:
        print("Warning: Multiple active gcloud accounts found. Using the first one.")
        user = user_output[0]
    else: # len(user_output) == 0
        print("Warning: No active gcloud account found. 'user' will be set to 'unknown'.")
        user = "unknown"

except KeyError:
    print("Warning: GOOGLE_CLOUD_PROJECT environment variable not found. 'project_id' will be set to 'unknown'.")
    project_id = "unknown"
    user = "unknown" # Also set user to unknown if project_id fails, as gcloud might also fail.
except Exception as e: # Catch other potential errors from gcloud command
    print(f"Warning: Could not retrieve gcloud user. Error: {e}. 'user' will be set to 'unknown'.")
    user = "unknown"

print("--- Environment and Execution Info ---")
print(f"Project ID: {project_id}")
print(f"User: {user}")
print(f"Location Setting: {location}")
print(f"Formatted Timestamp: {formatted_date}")
print("------------------------------------")

# -----------------------------------------------------------------------------
# ACTION REQUIRED: Update these GCS path variables with your specific bucket names,
# file names, and folder paths before running the script.
# -----------------------------------------------------------------------------

# --- Input Configuration ---
# GCS bucket where your source Excel file is located.
input_excel_bucket = 'excel_files_aw'  # PLEASE VERIFY this is your correct input bucket.

# Name of the Excel file in the input_excel_bucket.
excel_filename = 'CDP_Dummy_Data.xlsx'

# --- Output Configuration ---
# GCS bucket where the final CSV files will be copied.
output_csv_bucket = 'csv_files_aw'  # REPLACE THIS with your actual output bucket name.

# --- Temporary CSV Storage Configuration (within input_excel_bucket) ---
# Folder name within the 'input_excel_bucket' where CSVs will be generated temporarily
# before being copied to the 'output_csv_bucket'. Ensure this ends with a '/'.
initial_csv_output_folder_name = 'generated_csvs_temp/'

# --- Final CSV Destination Configuration (within output_csv_bucket) ---
# Optional: Folder name within the 'output_csv_bucket' where the final CSVs will be placed.
# Ensure this ends with a '/' if used. Use an empty string "" for the root of the output_csv_bucket.
destination_folder_in_output_bucket = 'final_cdp_csvs/'

# --- BigQuery Configuration ---
# The BigQuery Project ID (often the same as project_id detected above, but can be different).
# If it's the same, you can use: bigquery_project_id = project_id
bigquery_project_id = project_id # Or replace with a specific project ID if different.

# The BigQuery Dataset ID where tables will be created/loaded. This dataset must exist.
bigquery_dataset_id = 'excel_bq' # REPLACE THIS with your BigQuery Dataset ID.

# Optional: A prefix for your BigQuery table names.
# E.g., if 'excel_import_', a CSV 'my_data.csv' becomes table 'excel_import_my_data'.
# Use an empty string for no prefix.
bigquery_table_prefix = 'excel_import_'

# Number of leading rows to skip in the CSV files when loading into BigQuery.
# If your CSV has one header row, set this to 1.
# If it has (for example) a title row and then a header row, set this to 2.
# ACTION REQUIRED: Inspect your CSV files and set this value appropriately.
bigquery_skip_leading_rows = 1 # ADJUST AS NEEDED (e.g., 1 for standard header, 2 if more rows before header)

print("\n--- GCS Path Configuration Loaded ---")
print(f"Input Excel Bucket: {input_excel_bucket}")
print(f"Excel Filename: {excel_filename}")
print(f"Output CSV Bucket (Final Destination): {output_csv_bucket}")
print(f"Temporary CSV Folder (in Input Bucket): {initial_csv_output_folder_name}")
print(f"Destination Folder (in Output Bucket): {destination_folder_in_output_bucket if destination_folder_in_output_bucket else '(root)'}")
print("\n--- BigQuery Configuration Loaded ---")
print(f"BigQuery Project ID: {bigquery_project_id}")
print(f"BigQuery Dataset ID: {bigquery_dataset_id}")
print(f"BigQuery Table Prefix: {bigquery_table_prefix if bigquery_table_prefix else '(none)'}")
print(f"BigQuery Skip Leading Rows: {bigquery_skip_leading_rows}")
print("---------------------------------")
# --- End of Code Cell 1 ---

**Excel to csv Function**

In [ ]:
# Code Cell 2: Excel to CSV Conversion Function (GCS Only)
# This function takes an Excel file from GCS, reads each sheet,
# and saves it as a separate CSV file to a GCS location.

def sanitize_column_name(col_name):
    """Converts a column name to a BigQuery-compatible format."""
    if not isinstance(col_name, str):
        col_name = str(col_name)
    # Replace non-alphanumeric characters (excluding underscore) with underscore
    col_name = re.sub(r'[^\w_]', '_', col_name)
    # Remove leading/trailing underscores that might result from replacements
    col_name = col_name.strip('_')
    # If the name starts with a digit, prefix with an underscore
    if col_name and col_name[0].isdigit():
        col_name = '_' + col_name
    # Replace multiple underscores with a single underscore
    col_name = re.sub(r'_+', '_', col_name)
    # Convert to lowercase for consistency (optional, but good practice)
    col_name = col_name.lower()
    # BigQuery max column name length is 300
    return col_name[:300] if col_name else "unnamed_column"


def excel_to_csv_all_sheets_gcs_only(excel_gcs_filepath, output_gcs_dir):
    """
    Reads an Excel file from GCS and saves each sheet as a separate CSV file to GCS,
    with sanitized column names for BigQuery compatibility.

    Args:
        excel_gcs_filepath (str): The GCS path to the Excel file (e.g., "gs://bucket/file.xlsx").
        output_gcs_dir (str): The GCS directory where CSV files will be saved (e.g., "gs://bucket/output_csvs/").
                              Must end with a '/'.

    Returns:
        list: A list of GCS paths to the generated CSV files, or an empty list if errors occur.
    """
    generated_csv_paths = []  # To store paths of successfully created CSVs
    try:
        if not excel_gcs_filepath.startswith("gs://"):
            print(f"Error: excel_gcs_filepath '{excel_gcs_filepath}' must be a GCS path (starting with 'gs://').")
            return []
        if not output_gcs_dir.startswith("gs://"):
            print(f"Error: output_gcs_dir '{output_gcs_dir}' must be a GCS path (starting with 'gs://').")
            return []

        print(f"Reading Excel file from GCS: {excel_gcs_filepath}")
        xls = pd.ExcelFile(excel_gcs_filepath)
        sheet_names = xls.sheet_names

        if not output_gcs_dir.endswith('/'):
            output_gcs_dir += '/'
            print(f"Adjusted GCS output directory to: {output_gcs_dir}")

        print(f"Found sheets: {sheet_names}")
        base_filename = os.path.splitext(excel_gcs_filepath.split("/")[-1])[0]

        for sheet_name in sheet_names:
            print(f"Processing sheet: {sheet_name}...")
            df = pd.read_excel(xls, sheet_name=sheet_name)

            # Sanitize column names
            original_columns = list(df.columns)
            sanitized_columns = [sanitize_column_name(col) for col in original_columns]

            # Handle potential duplicate sanitized column names by appending a suffix
            final_columns = []
            counts = {}
            for col in sanitized_columns:
                if col in counts:
                    counts[col] += 1
                    final_columns.append(f"{col}_{counts[col]}")
                else:
                    counts[col] = 0 # Start count at 0 for first occurrence, suffix from _1
                    final_columns.append(col)

            df.columns = final_columns
            print(f"Sanitized column names for sheet '{sheet_name}': {list(df.columns)}")


            safe_sheet_name_for_filename = "".join(c if c.isalnum() else "_" for c in str(sheet_name))
            # Consistent naming with original Excel file and sheet name
            csv_filename = f"{base_filename} - {safe_sheet_name_for_filename}.csv"
            csv_gcs_filepath = f"{output_gcs_dir}{csv_filename}"

            print(f"Attempting to save CSV to GCS: {csv_gcs_filepath}")
            df.to_csv(csv_gcs_filepath, index=False, encoding='utf-8')
            generated_csv_paths.append(csv_gcs_filepath)
            print(f"Successfully converted sheet '{sheet_name}' to '{csv_gcs_filepath}'")

        return generated_csv_paths

    except FileNotFoundError:
        print(f"Error: Excel file not found at GCS path '{excel_gcs_filepath}'. Ensure the path is correct and permissions are set.")
        return []
    except Exception as e:
        print(f"An error occurred during Excel to CSV conversion (GCS only): {e}")
        if "No such object" in str(e) or "access denied" in str(e).lower() or "forbidden" in str(e).lower():
             print(f"Hint: This could be a GCS permission issue or the file '{excel_gcs_filepath}' truly doesn't exist.")
        elif "Could not connect to GCS" in str(e):
            print("Hint: You might need to authenticate for GCS access. Colab Enterprise usually handles this via its service account.")
        return []

print("GCS-only Excel to CSV conversion function defined: excel_to_csv_all_sheets_gcs_only")
# --- End of Code Cell 2 ---

**Function to copy csv to a different gcs bucket**

In [ ]:
# Code Cell 3: Function to Copy CSVs to a Different GCS Bucket
# This function copies .csv files from a source GCS location to a destination GCS bucket.
def copy_gcs_files_to_another_bucket(source_gcs_directory_or_files_list, dest_bucket_name, dest_folder_path=""):
    """
    Copies .csv files from a GCS directory or a list of GCS file paths to another GCS bucket.

    Args:
        source_gcs_directory_or_files_list (str or list): Source GCS path(s).
        dest_bucket_name (str): Destination GCS bucket name.
        dest_folder_path (str, optional): Folder within the destination bucket.
    Returns:
        list: A list of GCS paths to the successfully copied files in the destination.
    """
    copied_file_paths = []
    try:
        storage_client = storage.Client()

        if dest_folder_path and not dest_folder_path.endswith('/'):
            dest_folder_path += '/'
            print(f"Adjusted GCS destination folder to: {dest_folder_path}")

        print(f"Preparing to copy files to destination: gs://{dest_bucket_name}/{dest_folder_path}")
        source_files_to_copy = []

        if isinstance(source_gcs_directory_or_files_list, str):
            if not source_gcs_directory_or_files_list.startswith("gs://") or not source_gcs_directory_or_files_list.endswith("/"):
                print("Error: Source GCS directory string must start with 'gs://' and end with '/'.")
                print(f"DEBUG: copy_gcs_files_to_another_bucket is returning [] due to invalid source directory string.") # DEBUG
                return []

            source_bucket_name_from_dir = source_gcs_directory_or_files_list.split('/')[2]
            source_prefix_from_dir = '/'.join(source_gcs_directory_or_files_list.split('/')[3:])
            source_bucket_obj = storage_client.bucket(source_bucket_name_from_dir)
            blobs = source_bucket_obj.list_blobs(prefix=source_prefix_from_dir)

            for blob in blobs:
                if blob.name.endswith(".csv") and blob.name != source_prefix_from_dir:
                    source_files_to_copy.append(f"gs://{source_bucket_name_from_dir}/{blob.name}")
            if not source_files_to_copy:
                print(f"No .csv files found in source directory: {source_gcs_directory_or_files_list}")
                print(f"DEBUG: copy_gcs_files_to_another_bucket is returning [] because no CSVs found in source directory.") # DEBUG
                return []

        elif isinstance(source_gcs_directory_or_files_list, list):
            source_files_to_copy = [f for f in source_gcs_directory_or_files_list if f.startswith("gs://") and f.endswith(".csv")]
            if not source_files_to_copy:
                print("No valid GCS .csv file paths provided in the list for copying.")
                print(f"DEBUG: copy_gcs_files_to_another_bucket is returning [] because no valid GCS CSV paths in input list.") # DEBUG
                return []
        else:
            print("Error: source_gcs_directory_or_files_list must be a GCS path string or a list of GCS file paths.")
            print(f"DEBUG: copy_gcs_files_to_another_bucket is returning [] due to invalid input type for source_gcs_directory_or_files_list.") # DEBUG
            return []

        destination_bucket_obj = storage_client.bucket(dest_bucket_name)

        for source_file_gcs_path in source_files_to_copy:
            source_bucket_name = source_file_gcs_path.split('/')[2]
            source_blob_name = '/'.join(source_file_gcs_path.split('/')[3:])
            source_blob_obj = storage_client.bucket(source_bucket_name).blob(source_blob_name)
            file_name_only = source_blob_name.split('/')[-1]
            destination_blob_name = f"{dest_folder_path}{file_name_only}"

            print(f"Copying '{source_file_gcs_path}' to 'gs://{dest_bucket_name}/{destination_blob_name}'...")
            new_blob = source_blob_obj.bucket.copy_blob(
                source_blob_obj, destination_bucket_obj, destination_blob_name
            )
            copied_file_path = f"gs://{dest_bucket_name}/{new_blob.name}"
            print(f"Successfully copied to '{copied_file_path}'")
            copied_file_paths.append(copied_file_path)

        print(f"DEBUG: copy_gcs_files_to_another_bucket successfully populated and is about to return: {copied_file_paths}") # DEBUG
        return copied_file_paths

    except Exception as e:
        print(f"An error occurred during GCS copy: {e}")
        if "Could not connect to GCS" in str(e) or \
           "does not have storage.objects.get access" in str(e) or \
           "does not have storage.objects.create access" in str(e) or \
           "Forbidden" in str(e).lower():
             print("Hint: Check GCS bucket/object permissions for both source and destination.")
        print(f"DEBUG: copy_gcs_files_to_another_bucket is returning [] due to an exception during the copy process.") # DEBUG
        return []

print("GCS copy function defined: copy_gcs_files_to_another_bucket")
# --- End of Code Cell 3 ---

**Function to load CSVs from GCS to BigQuery using LOAD DATA SQL**

In [ ]:
# Code Cell 3.5: Function to Load CSVs from GCS to BigQuery using LOAD DATA SQL
# This function uses the BigQuery Python client to execute LOAD DATA SQL statements.

def load_csvs_to_bigquery_with_sql_loop(csv_gcs_uris, bq_project_id, bq_dataset_id,
                                        bq_table_prefix="", skip_leading_rows=1):
    """
    Loads CSV files from GCS into BigQuery tables using LOAD DATA SQL statements
    executed via the BigQuery Python client.

    Args:
        csv_gcs_uris (list): A list of GCS URIs for the CSV files to load.
        bq_project_id (str): The Google Cloud project ID where the BigQuery dataset resides.
        bq_dataset_id (str): The BigQuery dataset ID. This dataset must already exist.
        bq_table_prefix (str, optional): A prefix to add to each BigQuery table name.
        skip_leading_rows (int, optional): Number of header rows to skip in the CSV.
    """
    if not csv_gcs_uris:
        print("No CSV GCS URIs provided to load into BigQuery. Skipping.")
        return

    # Initialize BigQuery client
    # The client will use the project ID from the environment or the one specified.
    # It's good practice to specify the project for clarity if it might differ from the notebook's default.
    bq_client = bigquery.Client(project=bq_project_id)

    print(f"\n--- Starting BigQuery Load Process (using LOAD DATA SQL) ---")
    print(f"Target BigQuery Project: {bq_project_id}, Dataset: {bq_dataset_id}")

    for csv_uri in csv_gcs_uris:
        try:
            # Derive table name from CSV filename
            file_name_with_ext = csv_uri.split('/')[-1]
            table_name_base = os.path.splitext(file_name_with_ext)[0]
            # Sanitize table name: replace non-alphanumeric with underscore
            sanitized_table_name_base = "".join(c if c.isalnum() else "_" for c in table_name_base)
            # Ensure it doesn't start with a number and is valid
            if not sanitized_table_name_base or sanitized_table_name_base[0].isdigit():
                sanitized_table_name_base = "_" + sanitized_table_name_base

            table_name_in_sql = f"`{bq_project_id}.{bq_dataset_id}.{bq_table_prefix}{sanitized_table_name_base}`"

            # Construct the LOAD DATA SQL query
            # Schema is autodetected by default in LOAD DATA for CSV if not specified.
            # WRITE_TRUNCATE equivalent is LOAD DATA OVERWRITE
            sql_query = f"""
            LOAD DATA OVERWRITE {table_name_in_sql}
            FROM FILES (
                format = 'CSV',
                uris = ['{csv_uri}'],
                skip_leading_rows = {skip_leading_rows}
                -- Other options like field_delimiter, null_marker can be added here if needed
                -- e.g., field_delimiter = ';',
                --       null_marker = '\\N'
            );
            """

            print(f"Executing LOAD DATA SQL for '{csv_uri}' into table {table_name_in_sql}:")
            print(f"SQL Query:\n{sql_query}")

            # Execute the LOAD DATA query
            query_job = bq_client.query(sql_query)
            query_job.result()  # Waits for the job to complete.

            print(f"Successfully loaded '{csv_uri}' into BigQuery table {table_name_in_sql}.")

        except Exception as e:
            print(f"An error occurred while trying to load CSV '{csv_uri}' with LOAD DATA SQL: {e}")
            if hasattr(e, 'errors') and e.errors:
                for error_detail in e.errors: # Renamed variable for clarity
                    print(f"  BigQuery error detail: {error_detail['message']}")


print("BigQuery CSV loading function (using LOAD DATA SQL) defined: load_csvs_to_bigquery_with_sql_loop")

**Run the Process**

In [ ]:
# Code Cell 4: Execution - Run the Process
# This cell uses the configuration from Code Cell 1 to construct full GCS paths
# and then calls the functions to perform the Excel-to-CSV conversion, GCS copy, and BigQuery load.

# Note: Colab Enterprise typically uses the attached service account for GCS access.
# Ensure the service account for your Colab Enterprise instance has the necessary GCS and BigQuery permissions.
print("Proceeding with GCS and BigQuery operations using the Colab Enterprise environment's authentication.")

# --- Construct full GCS paths using variables from Code Cell 1 ---
gcs_excel_file_path = f"gs://{input_excel_bucket}/{excel_filename}"
initial_gcs_output_directory_for_csvs = f"gs://{input_excel_bucket}/{initial_csv_output_folder_name}"

# --- Execution ---
# It's assumed that configurations in Code Cell 1 are correctly set.
# Please ensure all variables in Code Cell 1 are verified before running this cell.

print(f"--- Execution Plan (using configuration from Code Cell 1) ---")
print(f"Input Excel File GCS Path: {gcs_excel_file_path}")
print(f"Initial CSVs Output (Temporary Location): {initial_gcs_output_directory_for_csvs}")
print(f"Final CSVs Destination Bucket: gs://{output_csv_bucket}/")
print(f"Final CSVs Destination Folder: {destination_folder_in_output_bucket if destination_folder_in_output_bucket else '(root of bucket)'}")
print(f"BigQuery Target Project: {bigquery_project_id}, Dataset: {bigquery_dataset_id}")
print(f"BigQuery Skip Leading Rows: {bigquery_skip_leading_rows}")
print(f"--------------------------------------------------------------")

# --- Step 1: Convert Excel to CSVs in the initial GCS location ---
print(f"\n--- Starting Excel to CSV Conversion (GCS Only) ---")
generated_csv_files_list_initial_loc = excel_to_csv_all_sheets_gcs_only(
    excel_gcs_filepath=gcs_excel_file_path,
    output_gcs_dir=initial_gcs_output_directory_for_csvs
)

if generated_csv_files_list_initial_loc:
    print(f"\n--- Excel to CSV Conversion Complete ---")
    print(f"Generated CSV files are temporarily located in: {initial_gcs_output_directory_for_csvs}")
    print(f"List of generated files: {generated_csv_files_list_initial_loc}")

    # --- Step 2: Copy generated CSVs to the separate output_csv_bucket ---
    print(f"\n--- Starting GCS Copy to Output Bucket ({output_csv_bucket}) ---")
    copied_csv_files_final_loc = copy_gcs_files_to_another_bucket(
        source_gcs_directory_or_files_list=generated_csv_files_list_initial_loc, # Use files from initial conversion
        dest_bucket_name=output_csv_bucket,
        dest_folder_path=destination_folder_in_output_bucket
    )

    if copied_csv_files_final_loc:
        print(f"\n--- GCS Copy Process Complete ---")
        final_destination_path_display = f"gs://{output_csv_bucket}/{destination_folder_in_output_bucket if destination_folder_in_output_bucket else ''}"
        print(f"Copied CSV files are now in: {final_destination_path_display}")
        print(f"List of copied files in final location: {copied_csv_files_final_loc}")

        # --- Step 3: Load CSVs from the final GCS location into BigQuery using LOAD DATA SQL ---
        load_csvs_to_bigquery_with_sql_loop( # Calling the new function
            csv_gcs_uris=copied_csv_files_final_loc,
            bq_project_id=bigquery_project_id,
            bq_dataset_id=bigquery_dataset_id,
            bq_table_prefix=bigquery_table_prefix,
            skip_leading_rows=bigquery_skip_leading_rows
        )
        print(f"\n--- BigQuery Load Process (using LOAD DATA SQL) Attempted ---")
        print(f"Check BigQuery dataset '{bigquery_project_id}.{bigquery_dataset_id}' for new tables.")

    else:
        print("\n--- GCS Copy did not produce any file paths. Skipping BigQuery load. ---")
else:
    print("\n--- Excel to CSV Conversion did not produce any files or encountered an error. Skipping subsequent steps. ---")
    print("   Please check the logs from the 'excel_to_csv_all_sheets_gcs_only' function for details.")

# --- End of Code Cell 4 ---